# ENARES 2024 CRS04 — Stage 03

## NB04 · Diccionario de indicadores, linaje y metadata técnica

Genera el diccionario curado de los indicadores traducidos en Stage 03, lo valida contra la tabla analítica real, registra el linaje SQL, inventaría los jobs de BigQuery y exporta evidencia en CSV, XLSX y Markdown.

In [1]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes openpyxl XlsxWriter tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.2 MB/s eta 0:00:00


In [2]:
from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import hashlib

auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = "enares-2024-crs04"
LOCATION = "US"
EXPECTED_ROWS = 18807

ROOT_DRIVE = Path("/content/drive/MyDrive/ENARES_2024_PROJECT")
LOG_DIR = ROOT_DRIVE / "05Resultados" / "logs" / "stage03"
SQL_DIR = ROOT_DRIVE / "02SQL"
OUTPUT_DIR = ROOT_DRIVE / "04Outputs"
DOCS_DIR = ROOT_DRIVE / "docs"

for d in [LOG_DIR, SQL_DIR, OUTPUT_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

A = f"{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents"

table = client.get_table(A)
if table.num_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"Row count inválido: {table.num_rows}; esperado: {EXPECTED_ROWS}"
    )

schema_df = pd.DataFrame([
    {
        "column_name": f.name,
        "data_type": f.field_type,
        "mode": f.mode,
        "description": f.description or "",
    }
    for f in table.schema
])

schema_df.to_csv(LOG_DIR / "stage3_schema_inventory.csv", index=False)

print("Tabla:", A)
print("Filas:", table.num_rows)
print("Columnas:", len(table.schema))

Mounted at /content/drive
Tabla: enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents
Filas: 18807
Columnas: 1405


## 1. Diccionario curado

El diccionario contiene los indicadores confirmados durante la traducción de los módulos 3.1–3.6, acumulación, consecuencias y riesgo/desprotección.

In [3]:
rows = []

def add(
    indicator_name,
    label,
    module,
    source_variables,
    numerator,
    denominator,
    missing_rule,
    allowed_values,
    source_spss,
    notebook,
    variable_type="binary indicator",
    status="implemented",
):
    rows.append({
        "indicator_name": indicator_name,
        "label": label,
        "module": module,
        "source_variables": source_variables,
        "numerator": numerator,
        "denominator": denominator,
        "missing_rule": missing_rule,
        "allowed_values": allowed_values,
        "source_spss": source_spss,
        "notebook": notebook,
        "variable_type": variable_type,
        "status": status,
    })

S31 = "07_CRS04_3.1_Caracteristicas_violencia_Percepciones_ver6.sps"
S32 = "08_CRS04_3.2 Violencia en el hogar_ver6.sps"
S33 = "09_CRS04_3.3 Violencia en el entorno escolar_ver4.sps"
S34 = "10_CRS04_3.4 Violencia sexual en adolescentes de 12 a 17 años_ver4.sps"
S35 = "11_CRS04_3.5 Acumulación de violencias_ver4.sps"
S36 = "13_CRS04_3.6_BusquedaAyuda_VS_ver4.sps"

# 3.1
add("justifica_castigo_docente","Justifies physical punishment by a teacher","3.1","C3P301_4","C3P301_4 = 1","C3P301_4 IN (1,2)","3 or NULL -> NULL","0,1,NULL",S31,"NB01")
add("justifica_castigo_parental","Justifies physical punishment by parents/caregivers","3.1","C3P301_5","C3P301_5 = 1","C3P301_5 IN (1,2)","3 or NULL -> NULL","0,1,NULL",S31,"NB01")
add("justifica_al_menos_una","Justifies at least one punishment form","3.1","justifica_castigo_docente;justifica_castigo_parental","At least one component = 1","At least one component observed","Both NULL -> NULL","0,1,NULL",S31,"NB01","binary composite indicator")
add("n_formas_justificadas","Number of punishment forms justified","3.1","justifica_castigo_docente;justifica_castigo_parental","SUM of valid components","At least one component observed","Both NULL -> NULL","0,1,2,NULL",S31,"NB01","count")
add("reconoce_derecho_opinar","Recognizes right to express an opinion","3.1","C3P301_2","C3P301_2 = 1","C3P301_2 IN (1,2)","3 or NULL -> NULL","0,1,NULL",S31,"NB01")
add("reconoce_derecho_denunciar","Recognizes right to report violence","3.1","C3P301_6","C3P301_6 = 1","C3P301_6 IN (1,2)","3 or NULL -> NULL","0,1,NULL",S31,"NB01")
add("rechaza_dejar_estudiar","Rejects preventing children from studying","3.1","C3P301_3","C3P301_3 = 2","C3P301_3 IN (1,2)","3 or NULL -> NULL","0,1,NULL",S31,"NB01")
add("rechaza_trabajo_infantil_necesidad","Rejects child labour as necessity","3.1","C3P301_1","C3P301_1 = 2","C3P301_1 IN (1,2)","3 or NULL -> NULL","0,1,NULL",S31,"NB01")
add("indice_derechos","Rights-recognition index","3.1","Four rights indicators","SUM of valid rights indicators","At least one component observed","All NULL -> NULL","0,1,2,3,4,NULL",S31,"NB01","count index")
add("reconoce_todos_derechos_clave","Recognizes all key rights","3.1","indice_derechos","indice_derechos = 4","indice_derechos observed","NULL -> NULL","0,1,NULL",S31,"NB01")
add("reconoce_3omas_derechos","Recognizes three or more rights","3.1","indice_derechos","indice_derechos >= 3","indice_derechos observed","NULL -> NULL","0,1,NULL",S31,"NB01")
add("predominio_femenino_tareas","Female predominance in household tasks","3.1","C3P302_1:C3P302_10","Female proportion > male proportion","Both proportions observed","Required value NULL -> NULL","0,1,NULL",S31,"NB01")
add("mito_locas","Believes myth that victims are mentally unstable","3.1","C3P303_1","C3P303_1 = 1","C3P303_1 IN (1,2)","3 or NULL -> NULL","0,1,NULL",S31,"NB01")
add("mito_pobreza","Believes myth linking sexual violence to poverty","3.1","C3P303_3","C3P303_3 = 1","C3P303_3 IN (1,2)","3 or NULL -> NULL","0,1,NULL",S31,"NB01")
add("mito_fuera_casa","Believes myth that sexual violence occurs outside home","3.1","C3P303_4","C3P303_4 = 1","C3P303_4 IN (1,2)","3 or NULL -> NULL","0,1,NULL",S31,"NB01")
add("mito_sitios_oscuros","Believes myth that sexual violence occurs in dark places","3.1","C3P303_5","C3P303_5 = 1","C3P303_5 IN (1,2)","3 or NULL -> NULL","0,1,NULL",S31,"NB01")
add("n_mitos","Number of sexual-violence myths believed","3.1","Four myth indicators","SUM of valid myth indicators","At least one component observed","All NULL -> NULL","0,1,2,3,4,NULL",S31,"NB01","count")
add("cree_al_menos_un_mito","Believes at least one sexual-violence myth","3.1","Four myth indicators","At least one component = 1","At least one component observed","All NULL -> NULL","0,1,NULL",S31,"NB01","binary composite indicator")
add("n_mitos_cat","Categorized number of myths","3.1","n_mitos","0->0; 1->1; 2-4->2","n_mitos observed","NULL -> NULL","0,1,2,NULL",S31,"NB01","categorical indicator")

# 3.2
add("VP_HOGAR","Psychological violence in household","3.2","C3P201_*;C3P203;aggressor variables","At least one qualifying psychological form","Valid household-violence universe","Translated SPSS filters/SYSMIS","0,1,NULL",S32,"NB02")
add("VF_HOGAR","Physical violence in household","3.2","C3P205_*;C3P207;aggressor variables","At least one qualifying physical form","Valid household-violence universe","Translated SPSS filters/SYSMIS","0,1,NULL",S32,"NB02")
add("VF_HOGAR_01","Household neglect through basic-needs deprivation","3.2","C3P121","Qualifying deprivation response","Valid C3P121 response","Translated SPSS rule","0,1,NULL",S32,"NB02")
add("VF_HOGAR_03","Household neglect through family unprotection","3.2","C3P216A_*;C3P216C_*","At least one unprotection situation","Applicable unprotection universe","Outside universe -> NULL","0,1,NULL",S32,"NB02")
add("VN_HOGAR1","Neglect in household","3.2","VF_HOGAR_01;VF_HOGAR_03","Any component = 1","At least one component observed","All NULL -> NULL","0,1,NULL",S32,"NB02","binary composite indicator")
add("INDICADOR_8_3_5","Any psychological/physical violence or neglect in household","3.2","VP_HOGAR;VF_HOGAR;VN_HOGAR1","Any component = 1","At least one component observed","All NULL -> NULL","0,1,NULL",S32,"NB02","binary composite indicator")

# Risk
add("DESP","Family-unprotection classification","Risk/unprotection","Translated risk components","SPSS-defined classification","Applicable risk universe","Outside universe -> NULL","SPSS-defined categories,NULL","14_CRS03_Ficha_Riesgo_Desproteccion_NN_9a11.sps","Risk notebook","risk classification")
add("rd12_idx_noviol","Twelve-month non-violence risk index","Risk/unprotection","Translated non-violence risk components","SUM of valid components","Applicable risk universe","Outside universe -> NULL","non-negative integer,NULL","Risk/desprotection syntax","Risk notebook","count index")
add("rd12_idx_noviol_cat","Categorized non-violence risk index","Risk/unprotection","rd12_idx_noviol","SPSS-defined categorization","rd12_idx_noviol observed","NULL -> NULL","SPSS-defined categories,NULL","Risk/desprotection syntax","Risk notebook","categorical index")

# 3.3
add("VP_ESCUELA","Psychological violence in school","3.3","C3P223_*;C3P225;aggressor variables","At least one qualifying psychological form","Valid school-violence universe","Translated SPSS filters/SYSMIS","0,1,NULL",S33,"NB03")
add("VF_ESCUELA","Physical violence in school","3.3","C3P227_*;C3P229;aggressor variables","At least one qualifying physical form","Valid school-violence universe","Translated SPSS filters/SYSMIS","0,1,NULL",S33,"NB03")

# 3.4
add("INDICADOR_8_3_6","Sexual violence indicator","3.4","C4P248_1:C4P248_16","At least one qualifying sexual-violence situation","Applicable CRS04 sexual-violence universe","Translated SPSS filters/SYSMIS","0,1,NULL",S34,"NB04")

# 3.5 accumulation
for name, label, src, rule in [
    ("PV_hogar_escuela1","Violence accumulation across household and school","Household and school violence indicators","SPSS-defined accumulation condition"),
    ("PV_hogar_escuela","Any violence in both household and school","Household and school indicators","Any household violence = 1 AND any school violence = 1"),
    ("PV_VP_hogar_escuela","Psychological violence in household and school","VP_HOGAR;VP_ESCUELA","VP_HOGAR = 1 AND VP_ESCUELA = 1"),
    ("PV_VF_hogar_escuela","Physical violence in household and school","VF_HOGAR;VF_ESCUELA","VF_HOGAR = 1 AND VF_ESCUELA = 1"),
    ("PV_VP_hogar_VF_escuela","Psychological household and physical school violence","VP_HOGAR;VF_ESCUELA","VP_HOGAR = 1 AND VF_ESCUELA = 1"),
    ("PV_VF_hogar_VP_escuela","Physical household and psychological school violence","VF_HOGAR;VP_ESCUELA","VF_HOGAR = 1 AND VP_ESCUELA = 1"),
    ("PV_VP_VF_hogar_escuela","Psychological and physical violence across household/school","VP_HOGAR;VF_HOGAR;VP_ESCUELA;VF_ESCUELA","SPSS-defined joint VP/VF condition"),
    ("PV_VP_VF_hogar_escuela_VS","Household, school and sexual polyvictimization","PV_VP_VF_hogar_escuela;INDICADOR_8_3_6","Both components = 1"),
]:
    add(name,label,"3.5",src,rule,"Required components observed","Insufficient component information -> NULL","0,1,NULL",S35,"NB06","binary composite indicator")

# 3.5.4 consequences
add("CONS_ALGUNA","At least one consequence","3.5.4","C3P243_1:C3P243_6","At least one valid item = 1","Applicable consequence universe","Outside universe -> NULL","0,1,NULL","Translated consequence syntax","NB07","binary composite indicator")
add("CONS_NUM_CONSECUENCIAS","Number of consequences","3.5.4","C3P243_1:C3P243_6","COUNT/SUM of items = 1","Applicable consequence universe","Outside universe -> NULL","0-6,NULL","Translated consequence syntax","NB07","count")
add("CONS_ATENCION_SALUD","Health attention due to consequences","3.5.4","Consequence and health-attention items","Health-attention condition satisfied","CONS_ALGUNA = 1","CONS_ALGUNA != 1 -> NULL","0,1,NULL","Translated consequence syntax","NB07","binary conditional indicator")

# 3.6 general
add("busco_ayuda_vs","Sought help after sexual violence","3.6","Help-seeking gateway/items","At least one help source selected","Applicable sexual-violence help universe","Outside universe -> NULL","0,1,NULL",S36,"NB08")
add("recibio_ayuda_vs","Received help after sexual violence","3.6","Help-received items","At least one qualifying help type = 1","Applicable help-seeking universe","Outside universe -> NULL","0,1,NULL",S36,"NB08")
add("recibio_ayuda_vs_victimas","Received help among victims","3.6","Victim indicator;recibio_ayuda_vs","recibio_ayuda_vs = 1","SPSS victim universe","Outside victim universe -> NULL","0,1,NULL",S36,"NB08","binary conditional indicator")
add("brecha_ayuda_vs","Help gap after sexual violence","3.6","busco_ayuda_vs;recibio_ayuda_vs","Help sought but not received","Applicable help universe","Outside universe -> NULL","0,1,NULL",S36,"NB08","binary gap indicator")
add("apoyo_institucional_vs","Sought institutional support","3.6","Institutional-source items","At least one institutional source selected","Applicable help universe","Outside universe -> NULL","0,1,NULL",S36,"NB08")
add("recibio_ayuda_institucional_vs","Received institutional help","3.6","Institutional-help items","At least one institutional-help item = 1","Applicable institutional universe","Outside universe -> NULL","0,1,NULL",S36,"NB08")
add("brecha_institucional_vs","Institutional-help gap","3.6","apoyo_institucional_vs;recibio_ayuda_institucional_vs","Support sought but help not received","Applicable institutional universe","Outside universe -> NULL","0,1,NULL",S36,"NB08","binary gap indicator")
add("conoce_demuna","Knows DEMUNA","3.6","DEMUNA awareness item","Yes response","Valid awareness responses","Invalid/not applicable -> NULL","0,1,NULL",S36,"NB08")
add("uso_demuna","Used DEMUNA","3.6","DEMUNA use item","Reported use","Applicable DEMUNA universe","Outside universe -> NULL","0,1,NULL",S36,"NB08")

help_vars = {
    "ayuda_vs_familiar":"Sought help from family",
    "ayuda_vs_madre":"Sought help from mother",
    "ayuda_vs_padre":"Sought help from father",
    "ayuda_vs_madrastra":"Sought help from stepmother",
    "ayuda_vs_padrastro":"Sought help from stepfather",
    "ayuda_vs_hermana":"Sought help from sister",
    "ayuda_vs_hermano":"Sought help from brother",
    "ayuda_vs_abuela":"Sought help from grandmother",
    "ayuda_vs_abuelo":"Sought help from grandfather",
    "ayuda_vs_tia":"Sought help from aunt",
    "ayuda_vs_tio":"Sought help from uncle",
    "ayuda_vs_otro_pariente":"Sought help from another relative",
    "ayuda_vs_car":"Sought help from residential-care service",
    "ayuda_vs_escolar_adulto":"Sought help from an adult at school",
    "ayuda_vs_pares_amigos":"Sought help from peers/friends",
    "ayuda_vs_otro":"Sought help from another source",
    "ayuda_vs_consejo":"Received advice",
    "ayuda_vs_hablo_madre_padre":"Someone spoke with mother/father",
    "ayuda_vs_reclamo_agresor":"Someone confronted the aggressor",
    "ayuda_vs_aviso_autoridades":"Authorities were notified",
    "ayuda_vs_refugio":"Received shelter/refuge",
    "ayuda_vs_especialista":"Received specialist support",
    "ayuda_vs_otro_tipo":"Received another help type",
    "ayuda_inst_vs_hablo_familia":"Institution spoke with family",
    "ayuda_inst_vs_terapias":"Institution provided/referred to therapy",
    "ayuda_inst_vs_llamo_atencion":"Institution called aggressor's attention",
    "ayuda_inst_vs_otro":"Institution provided another response",
}

for name, label in help_vars.items():
    add(
        name,label,"3.6",
        "Corresponding SPSS help item",
        "Corresponding item selected/yes",
        "Applicable sexual-violence help universe",
        "Within universe follow SPSS recode; outside universe -> NULL",
        "0,1,NULL",S36,"NB08","binary component indicator"
    )

dic = pd.DataFrame(rows).drop_duplicates("indicator_name", keep="last")
dic = dic.sort_values(["module","indicator_name"]).reset_index(drop=True)

actual_columns = set(schema_df["column_name"])
dic["exists_in_analytical"] = dic["indicator_name"].isin(actual_columns)
dic["validation_status"] = dic["exists_in_analytical"].map({
    True: "verified in analytical table",
    False: "not found - verify exact translated column name",
})

dic.to_excel(OUTPUT_DIR / "diccionario_indicadores.xlsx", index=False)
dic.to_csv(OUTPUT_DIR / "diccionario_indicadores.csv", index=False)

(DOCS_DIR / "stage3_data_dictionary.md").write_text(
    "# ENARES 2024 CRS04 - Stage 03 Data Dictionary\n\n"
    + dic.to_markdown(index=False),
    encoding="utf-8",
)

display(dic)
print("Dictionary rows:", len(dic))
print("Verified:", int(dic["exists_in_analytical"].sum()))
print("Not found:", int((~dic["exists_in_analytical"]).sum()))

,indicator_name,label,module,source_variables,numerator,denominator,missing_rule,allowed_values,source_spss,notebook,variable_type,status,exists_in_analytical,validation_status
0,cree_al_menos_un_mito,Believes at least one sexual-violence myth,3.1,Four myth indicators,At least one component = 1,At least one component observed,All NULL -> NULL,"0,1,NULL",07_CRS04_3.1_Caracteristicas_violencia_Percepc...,NB01,binary composite indicator,implemented,True,verified in analytical table
1,indice_derechos,Rights-recognition index,3.1,Four rights indicators,SUM of valid rights indicators,At least one component observed,All NULL -> NULL,"0,1,2,3,4,NULL",07_CRS04_3.1_Caracteristicas_violencia_Percepc...,NB01,count index,implemented,True,verified in analytical table
2,justifica_al_menos_una,Justifies at least one punishment form,3.1,justifica_castigo_docente;justifica_castigo_pa...,At least one component = 1,At least one component observed,Both NULL -> NULL,"0,1,NULL",07_CRS04_3.1_Caracteristicas_violencia_Percepc...,NB01,binary composite indicator,implemented,True,verified in analytical table
3,justifica_castigo_docente,Justifies physical punishment by a teacher,3.1,C3P301_4,C3P301_4 = 1,"C3P301_4 IN (1,2)",3 or NULL -> NULL,"0,1,NULL",07_CRS04_3.1_Caracteristicas_violencia_Percepc...,NB01,binary indicator,implemented,True,verified in analytical table
4,justifica_castigo_parental,Justifies physical punishment by parents/careg...,3.1,C3P301_5,C3P301_5 = 1,"C3P301_5 IN (1,2)",3 or NULL -> NULL,"0,1,NULL",07_CRS04_3.1_Caracteristicas_violencia_Percepc...,NB01,binary indicator,implemented,True,verified in analytical table
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,recibio_ayuda_vs_victimas,Received help among victims,3.6,Victim indicator;recibio_ayuda_vs,recibio_ayuda_vs = 1,SPSS victim universe,Outside victim universe -> NULL,"0,1,NULL",13_CRS04_3.6_BusquedaAyuda_VS_ver4.sps,NB08,binary conditional indicator,implemented,True,verified in analytical table
74,uso_demuna,Used DEMUNA,3.6,DEMUNA use item,Reported use,Applicable DEMUNA universe,Outside universe -> NULL,"0,1,NULL",13_CRS04_3.6_BusquedaAyuda_VS_ver4.sps,NB08,binary indicator,implemented,True,verified in analytical table
75,DESP,Family-unprotection classification,Risk/unprotection,Translated risk components,SPSS-defined classification,Applicable risk universe,Outside universe -> NULL,"SPSS-defined categories,NULL",14_CRS03_Ficha_Riesgo_Desproteccion_NN_9a11.sps,Risk notebook,risk classification,implemented,False,not found - verify exact translated column name
76,rd12_idx_noviol,Twelve-month non-violence risk index,Risk/unprotection,Translated non-violence risk components,SUM of valid components,Applicable risk universe,Outside universe -> NULL,"non-negative integer,NULL",Risk/desprotection syntax,Risk notebook,count index,implemented,True,verified in analytical table


Dictionary rows: 78
Verified: 75
Not found: 3


## 2. Validación del diccionario contra la tabla analítica

In [4]:
validation = dic[
    [
        "indicator_name",
        "module",
        "notebook",
        "status",
        "exists_in_analytical",
        "validation_status",
    ]
].copy()

validation.to_csv(
    LOG_DIR / "stage3_dictionary_validation.csv",
    index=False,
)

missing = validation.loc[
    ~validation["exists_in_analytical"]
].copy()

missing.to_csv(
    LOG_DIR / "stage3_documented_but_missing.csv",
    index=False,
)

documented = set(dic["indicator_name"])
present_but_undocumented = schema_df.loc[
    ~schema_df["column_name"].isin(documented)
].copy()

present_but_undocumented.to_csv(
    LOG_DIR / "stage3_present_but_undocumented.csv",
    index=False,
)

display(validation)
display(missing)

,indicator_name,module,notebook,status,exists_in_analytical,validation_status
0,cree_al_menos_un_mito,3.1,NB01,implemented,True,verified in analytical table
1,indice_derechos,3.1,NB01,implemented,True,verified in analytical table
2,justifica_al_menos_una,3.1,NB01,implemented,True,verified in analytical table
3,justifica_castigo_docente,3.1,NB01,implemented,True,verified in analytical table
4,justifica_castigo_parental,3.1,NB01,implemented,True,verified in analytical table
...,...,...,...,...,...,...
73,recibio_ayuda_vs_victimas,3.6,NB08,implemented,True,verified in analytical table
74,uso_demuna,3.6,NB08,implemented,True,verified in analytical table
75,DESP,Risk/unprotection,Risk notebook,implemented,False,not found - verify exact translated column name
76,rd12_idx_noviol,Risk/unprotection,Risk notebook,implemented,True,verified in analytical table


,indicator_name,module,notebook,status,exists_in_analytical,validation_status
9,n_formas_justificadas,3.1,NB01,implemented,False,not found - verify exact translated column name
19,INDICADOR_8_3_5,3.2,NB02,implemented,False,not found - verify exact translated column name
75,DESP,Risk/unprotection,Risk notebook,implemented,False,not found - verify exact translated column name


## 3. Inventario y linaje SQL

In [5]:
sql_files = sorted(SQL_DIR.glob("*.sql"))

inventory_rows = []
lineage_rows = []

for path in sql_files:
    text = path.read_text(encoding="utf-8")
    sha = hashlib.sha256(text.encode("utf-8")).hexdigest()

    inventory_rows.append({
        "sql_file": path.name,
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sql_sha256": sha,
        "modified_utc": datetime.fromtimestamp(
            path.stat().st_mtime,
            tz=timezone.utc,
        ).isoformat(),
    })

    lineage_rows.append({
        "output_table": "analytical_crs04_adolescents",
        "source_tables": (
            "cleaned_crs04_merged_adolescents;"
            "prior analytical_crs04_adolescents state"
        ),
        "sql_file": path.name,
        "sql_sha256": sha,
        "run_utc": RUN_UTC,
    })

sql_inventory = pd.DataFrame(inventory_rows)
lineage = pd.DataFrame(lineage_rows)

if sql_inventory.empty:
    sql_inventory = pd.DataFrame(columns=[
        "sql_file","path","size_bytes","sql_sha256","modified_utc"
    ])

if lineage.empty:
    lineage = pd.DataFrame(columns=[
        "output_table","source_tables","sql_file","sql_sha256","run_utc"
    ])

sql_inventory.to_csv(LOG_DIR / "stage3_sql_inventory.csv", index=False)
lineage.to_csv(LOG_DIR / "stage3_lineage.csv", index=False)

display(sql_inventory)
display(lineage)

,sql_file,path,size_bytes,sql_sha256,modified_utc
0,stage3_35_4_consecuencias.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1454,0bcd44e7a15bbb206fe718b98bd123677ad8c8bb7bf62b...,2026-07-17T04:56:46+00:00
1,stage3_35_acumulacion_violencias.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1005,ef2085ea419a92d02e9adfb7dfadb796ca4e19d12085f5...,2026-07-17T04:41:06+00:00
2,stage3_36_busqueda_ayuda_vs.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,4209,53001119320403ccfdc0a772e66d730ce588cfc1f54acd...,2026-07-17T05:11:44+00:00
3,stage3_create_crs04_analytical.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1261,9e7bc4a9d1be612ea8f0e73a3522dfb3b316856a25ecf5...,2026-07-17T01:45:26+00:00
4,stage3_create_crs04_cleaned.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1518,3109ccc786c87cd6e30389bf9f968ecce36ca508d0a034...,2026-07-16T07:14:51+00:00
5,stage3_ficha_riesgo_desproteccion_crs04.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,16655,28a8e7d26f32af4ec77cb03ca2cdad9a4a3fa40a3690e8...,2026-07-17T04:23:51+00:00
6,stage3_master.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1261,9e7bc4a9d1be612ea8f0e73a3522dfb3b316856a25ecf5...,2026-07-17T01:45:26+00:00
7,stage3_syntax_31_actitudes.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1761,d7cc5505e730e6393d53983eeb29407f30bcf8402a25b0...,2026-07-17T01:45:47+00:00
8,stage3_syntax_31_derechos.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,2509,b57c2ee2143f993685154f0c2bb75709ebf60c826725d1...,2026-07-17T01:46:06+00:00
9,stage3_syntax_31_desagregaciones.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1502,41fb6ab9b9314d847c82aaeb5d3b96920687ea387de91a...,2026-07-17T01:45:40+00:00


,output_table,source_tables,sql_file,sql_sha256,run_utc
0,analytical_crs04_adolescents,cleaned_crs04_merged_adolescents;prior analyti...,stage3_35_4_consecuencias.sql,0bcd44e7a15bbb206fe718b98bd123677ad8c8bb7bf62b...,2026-07-17T07:28:29.709895+00:00
1,analytical_crs04_adolescents,cleaned_crs04_merged_adolescents;prior analyti...,stage3_35_acumulacion_violencias.sql,ef2085ea419a92d02e9adfb7dfadb796ca4e19d12085f5...,2026-07-17T07:28:29.709895+00:00
2,analytical_crs04_adolescents,cleaned_crs04_merged_adolescents;prior analyti...,stage3_36_busqueda_ayuda_vs.sql,53001119320403ccfdc0a772e66d730ce588cfc1f54acd...,2026-07-17T07:28:29.709895+00:00
3,analytical_crs04_adolescents,cleaned_crs04_merged_adolescents;prior analyti...,stage3_create_crs04_analytical.sql,9e7bc4a9d1be612ea8f0e73a3522dfb3b316856a25ecf5...,2026-07-17T07:28:29.709895+00:00
4,analytical_crs04_adolescents,cleaned_crs04_merged_adolescents;prior analyti...,stage3_create_crs04_cleaned.sql,3109ccc786c87cd6e30389bf9f968ecce36ca508d0a034...,2026-07-17T07:28:29.709895+00:00
5,analytical_crs04_adolescents,cleaned_crs04_merged_adolescents;prior analyti...,stage3_ficha_riesgo_desproteccion_crs04.sql,28a8e7d26f32af4ec77cb03ca2cdad9a4a3fa40a3690e8...,2026-07-17T07:28:29.709895+00:00
6,analytical_crs04_adolescents,cleaned_crs04_merged_adolescents;prior analyti...,stage3_master.sql,9e7bc4a9d1be612ea8f0e73a3522dfb3b316856a25ecf5...,2026-07-17T07:28:29.709895+00:00
7,analytical_crs04_adolescents,cleaned_crs04_merged_adolescents;prior analyti...,stage3_syntax_31_actitudes.sql,d7cc5505e730e6393d53983eeb29407f30bcf8402a25b0...,2026-07-17T07:28:29.709895+00:00
8,analytical_crs04_adolescents,cleaned_crs04_merged_adolescents;prior analyti...,stage3_syntax_31_derechos.sql,b57c2ee2143f993685154f0c2bb75709ebf60c826725d1...,2026-07-17T07:28:29.709895+00:00
9,analytical_crs04_adolescents,cleaned_crs04_merged_adolescents;prior analyti...,stage3_syntax_31_desagregaciones.sql,41fb6ab9b9314d847c82aaeb5d3b96920687ea387de91a...,2026-07-17T07:28:29.709895+00:00


## 4. Métricas de jobs de BigQuery

In [6]:
jobs_sql = f"""
SELECT
  job_id,
  statement_type,
  creation_time,
  start_time,
  end_time,
  TIMESTAMP_DIFF(end_time, start_time, MILLISECOND) AS duration_ms,
  total_bytes_processed,
  total_slot_ms
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE project_id = '{PROJECT_ID}'
  AND creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND state = 'DONE'
ORDER BY creation_time DESC
"""

jobs = client.query(jobs_sql, location=LOCATION).result().to_dataframe()
jobs.to_csv(LOG_DIR / "stage3_etl_jobs_metrics.csv", index=False)
display(jobs.head(20))

,job_id,statement_type,creation_time,start_time,end_time,duration_ms,total_bytes_processed,total_slot_ms
0,8a8c2d9a-9d17-4ae8-8276-fc23b60cb742,SELECT,2026-07-17 05:12:00.177000+00:00,2026-07-17 05:12:00.295000+00:00,2026-07-17 05:12:00.582000+00:00,287,226600,32
1,6a083661-709a-42c0-a543-9d66c0da2717,SELECT,2026-07-17 05:11:58.353000+00:00,2026-07-17 05:11:58.420000+00:00,2026-07-17 05:11:58.677000+00:00,257,264376,17
2,bb376851-d9d9-4637-8b42-a6dab17df72a,SELECT,2026-07-17 05:11:56.116000+00:00,2026-07-17 05:11:56.200000+00:00,2026-07-17 05:11:57.027000+00:00,827,2954248,176
3,93bffe4e-5934-448a-a920-dd2e69181a81,SELECT,2026-07-17 05:11:54.370000+00:00,2026-07-17 05:11:54.480000+00:00,2026-07-17 05:11:54.791000+00:00,311,908048,33
4,a557e324-7f26-429b-8e35-08328ff88c72,SELECT,2026-07-17 05:11:51.915000+00:00,2026-07-17 05:11:52.119000+00:00,2026-07-17 05:11:53.042000+00:00,923,2954248,187
5,61f22969-aea8-4eb2-b173-7c1e6c6299f6,CREATE_TABLE_AS_SELECT,2026-07-17 05:11:44.104000+00:00,2026-07-17 05:11:44.436000+00:00,2026-07-17 05:11:51.158000+00:00,6722,70984226,13206
6,15cb9e40-c366-4d49-ae2d-e687652768df,SELECT,2026-07-17 05:11:41.203000+00:00,2026-07-17 05:11:41.319000+00:00,2026-07-17 05:11:41.650000+00:00,331,488024,54
7,e053021e-39cc-42e0-a3bc-32ed7806da40,SELECT,2026-07-17 04:56:57.575000+00:00,2026-07-17 04:56:57.699000+00:00,2026-07-17 04:56:57.935000+00:00,236,116144,12
8,54a596a5-8bab-4352-a80b-805fd39c5863,SELECT,2026-07-17 04:56:55.897000+00:00,2026-07-17 04:56:56.074000+00:00,2026-07-17 04:56:56.299000+00:00,225,31480,18
9,cf270890-77ad-46e6-9c52-29233ad06cb3,SELECT,2026-07-17 04:56:54.337000+00:00,2026-07-17 04:56:54.423000+00:00,2026-07-17 04:56:54.635000+00:00,212,84664,21


## 5. Reporte técnico y cierre

In [7]:
summary = pd.DataFrame([{
    "run_utc": RUN_UTC,
    "analytical_table": A,
    "rows": table.num_rows,
    "columns": len(table.schema),
    "dictionary_rows": len(dic),
    "dictionary_verified": int(dic["exists_in_analytical"].sum()),
    "dictionary_missing": int((~dic["exists_in_analytical"]).sum()),
    "sql_files": len(sql_inventory),
    "job_records": len(jobs),
}])

summary.to_csv(LOG_DIR / "stage3_metadata_report.csv", index=False)

report_lines = [
    "# ENARES 2024 CRS04 - Stage 03 Metadata Report",
    "",
    f"- Run UTC: {RUN_UTC}",
    f"- Analytical table: `{A}`",
    f"- Rows: {table.num_rows}",
    f"- Columns: {len(table.schema)}",
    f"- Dictionary rows: {len(dic)}",
    f"- Verified dictionary indicators: {int(dic['exists_in_analytical'].sum())}",
    f"- Documented but missing: {int((~dic['exists_in_analytical']).sum())}",
    f"- SQL files inventoried: {len(sql_inventory)}",
    f"- BigQuery jobs inventoried: {len(jobs)}",
]

(DOCS_DIR / "stage3_metadata_report.md").write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)

required = [
    OUTPUT_DIR / "diccionario_indicadores.xlsx",
    OUTPUT_DIR / "diccionario_indicadores.csv",
    DOCS_DIR / "stage3_data_dictionary.md",
    DOCS_DIR / "stage3_metadata_report.md",
    LOG_DIR / "stage3_schema_inventory.csv",
    LOG_DIR / "stage3_dictionary_validation.csv",
    LOG_DIR / "stage3_documented_but_missing.csv",
    LOG_DIR / "stage3_present_but_undocumented.csv",
    LOG_DIR / "stage3_sql_inventory.csv",
    LOG_DIR / "stage3_lineage.csv",
    LOG_DIR / "stage3_etl_jobs_metrics.csv",
    LOG_DIR / "stage3_metadata_report.csv",
]

closure = pd.DataFrame({
    "path": [str(p) for p in required],
    "exists": [p.exists() for p in required],
    "run_utc": RUN_UTC,
})

closure.to_csv(
    LOG_DIR / "stage3_dictionary_lineage_closure.csv",
    index=False,
)

display(summary)
display(closure)

passed = table.num_rows == EXPECTED_ROWS and closure["exists"].all()
print("PASS" if passed else "NOT PASSED")

,run_utc,analytical_table,rows,columns,dictionary_rows,dictionary_verified,dictionary_missing,sql_files,job_records
0,2026-07-17T07:28:29.709895+00:00,enares-2024-crs04.enares2024_crs04_analytical....,18807,1405,78,75,3,31,317


,path,exists,run_utc
0,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...,True,2026-07-17T07:28:29.709895+00:00
1,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...,True,2026-07-17T07:28:29.709895+00:00
2,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,True,2026-07-17T07:28:29.709895+00:00
3,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,True,2026-07-17T07:28:29.709895+00:00
4,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,2026-07-17T07:28:29.709895+00:00
5,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,2026-07-17T07:28:29.709895+00:00
6,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,2026-07-17T07:28:29.709895+00:00
7,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,2026-07-17T07:28:29.709895+00:00
8,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,2026-07-17T07:28:29.709895+00:00
9,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,2026-07-17T07:28:29.709895+00:00


PASS
